# 07 — Inference: test-time augmentation, ensembling and the final submission

1. For every run in `RUNS` and every finished fold, predictions on val_clean, val_shift and test are loaded —
   with TTA they are computed once and cached in `outputs/preds/<run>/fold*_*_tta.npz`.
   TTA = average of the logits over framings {full image, 90 % centre crop, aspect × 0.85, aspect × 1.15}
   × {original, horizontally mirrored} (`src/transforms.py::apply_framing`).
2. Optional TTA ablation on one fold: accuracy of each framing on its own.
3. Ensemble: test probabilities are averaged over the folds of each run, then combined across runs with weights
   chosen by grid search (step 0.1) to maximise OOF **val_shift** accuracy.
4. The submission is written to `submissions/`, validated against `sample_submission.csv` and logged.

TTA on test is 8 forward passes over 29,443 images per fold model (ConvNeXt-Tiny at 128 px: roughly 10 minutes per fold
on the M1 Pro).

In [ ]:
RUNS = "convnext_tiny_shift"                 # comma-separated runs to ensemble
FOLDS = "all"                                # "all" = every finished fold of each run
TTA = True
FRAMINGS = "full,center90,narrow,wide"
TTA_ABLATION = True                          # per-framing accuracy on one fold of the first run
SUB_NAME = "auto"                            # file name (without .csv); "auto" builds one from the runs

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))  # the notebooks live in the repo root, next to src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

from src.config import CKPT_DIR, CLASSES, SUBMISSIONS_CSV, get_device, load_shiftsim_params, make_dirs
from src.data import load_folds_df, load_test_df
from src.engine import get_preds, load_trained_model, predict_view
from src.plots import savefig
from src.transforms import ShiftSim
from src.utils import accuracy, fit_ensemble_weights, log_loss, softmax, write_submission

make_dirs()
runs = [r.strip() for r in RUNS.split(",") if r.strip()]
framings = [f.strip() for f in FRAMINGS.split(",")]


def finished_folds(run):
    import torch
    out = []
    for d in sorted((CKPT_DIR / run).glob("fold*")):
        if (d / "best.pt").exists() and torch.load(d / "last.pt", map_location="cpu", weights_only=False)["finished"]:
            out.append(int(d.name[4:]))
    return out


run_folds = {r: (finished_folds(r) if FOLDS == "all" else [int(f) for f in FOLDS.split(",")]) for r in runs}
print(run_folds)
test_df = load_test_df()

## 1. Predictions per run and fold (with and without TTA)

In [ ]:
preds = {}   # (run, tta) -> dict(split -> (ids, probs, labels)); val splits are OOF over the run's folds
for run, folds in run_folds.items():
    for tta in sorted({False, TTA}):
        out = {}
        for split in ["val_clean", "val_shift"]:
            parts = [get_preds(run, f, split, tta=tta, framings=framings) for f in folds]
            out[split] = (np.concatenate([p[0] for p in parts]), softmax(np.concatenate([p[1] for p in parts])),
                          np.concatenate([p[2] for p in parts]))
        test_parts = [get_preds(run, f, "test", tta=tta, framings=framings) for f in folds]
        assert all((p[0] == test_df["image_id"].values).all() for p in test_parts)
        out["test"] = np.mean([softmax(p[1]) for p in test_parts], axis=0)
        preds[(run, tta)] = out

rows = []
for (run, tta), out in preds.items():
    rows.append({"run": run, "TTA": tta, "folds": len(run_folds[run]),
                 "OOF val_clean acc": accuracy(out["val_clean"][1], out["val_clean"][2]),
                 "OOF val_shift acc": accuracy(out["val_shift"][1], out["val_shift"][2]),
                 "OOF val_shift log loss": log_loss(out["val_shift"][1], out["val_shift"][2])})
display(pd.DataFrame(rows).round(4))

> **Observations (to be written by me):**

## 2. TTA ablation (one fold of the first run)

Accuracy of every single framing, with and without mirroring, and of the full TTA average.

In [ ]:
if TTA_ABLATION:
    run, fold = runs[0], run_folds[runs[0]][0]
    model, cfg = load_trained_model(run, fold)
    df = load_folds_df()
    df = df[df["fold"] == fold].reset_index(drop=True)
    sim = ShiftSim(**load_shiftsim_params())
    abl = []
    for framing in framings:
        for flip in [False, True]:
            row = {"framing": framing, "mirrored": flip}
            for split, s in [("val_clean", None), ("val_shift", sim)]:
                row[f"{split} acc"] = accuracy(predict_view(model, cfg, df, framing, flip, s), df["label"].values)
            abl.append(row)
    full = {"framing": "TTA average (all views)", "mirrored": "both"}
    for split in ["val_clean", "val_shift"]:
        _, logits, labels = get_preds(run, fold, split, tta=True, framings=framings)
        full[f"{split} acc"] = accuracy(logits, labels)
    abl.append(full)
    print(f"run {run}, fold {fold}")
    display(pd.DataFrame(abl).round(4))

> **Observations (to be written by me):**

## 3. Ensemble across runs

Weights are fitted on the OOF val_shift probabilities of the images that every run has predicted, and then used for
the test probabilities. The same weights are also applied to val_clean. With a single run the weight is 1.

In [ ]:
use_tta = TTA
common = None
for run in runs:
    ids = preds[(run, use_tta)]["val_shift"][0]
    common = set(ids) if common is None else common & set(ids)
common = np.array(sorted(common))
print("validation images predicted by every run:", len(common))


def aligned(run, split):
    ids, probs, labels = preds[(run, use_tta)][split]
    order = pd.Series(np.arange(len(ids)), index=ids).loc[common].values
    return probs[order], labels[order]


val_shift_probs = [aligned(r, "val_shift")[0] for r in runs]
labels = aligned(runs[0], "val_shift")[1]
weights, ens_acc = fit_ensemble_weights(val_shift_probs, labels)
val_clean_probs = [aligned(r, "val_clean")[0] for r in runs]
ens_clean = sum(w * p for w, p in zip(weights, val_clean_probs))
ens_shift = sum(w * p for w, p in zip(weights, val_shift_probs))

table = pd.DataFrame({"weight": weights,
                      "val_clean acc (common images)": [accuracy(p, labels) for p in val_clean_probs],
                      "val_shift acc (common images)": [accuracy(p, labels) for p in val_shift_probs]}, index=runs)
table.loc["ENSEMBLE"] = [1.0, accuracy(ens_clean, labels), accuracy(ens_shift, labels)]
display(table.round(4))

> **Observations (to be written by me):**

## 4. Final submission

In [ ]:
test_probs = sum(w * preds[(r, use_tta)]["test"] for w, r in zip(weights, runs))
if SUB_NAME == "auto":
    SUB_NAME = "ens_" + "+".join(f"{r}x{len(run_folds[r])}" for r in runs) + ("_tta" if use_tta else "")
write_submission(test_df, test_probs, SUB_NAME, runs="+".join(runs),
                 val_clean=round(accuracy(ens_clean, labels), 4), val_shift=round(accuracy(ens_shift, labels), 4),
                 notes=f"weights { {r: round(float(w), 2) for r, w in zip(runs, weights)} }, folds {run_folds}, TTA={use_tta}")

In [ ]:
train_prior = load_folds_df()["class"].value_counts(normalize=True).reindex(CLASSES)
pred_dist = pd.Series(np.array(CLASSES)[test_probs.argmax(1)]).value_counts(normalize=True).reindex(CLASSES)
display(pd.DataFrame({"train share": train_prior, "predicted test share": pred_dist}).round(4))
for r in runs:
    agree = (preds[(r, use_tta)]["test"].argmax(1) == test_probs.argmax(1)).mean()
    print(f"agreement of {r} with the final submission on test: {agree:.4f}")
display(pd.read_csv(SUBMISSIONS_CSV))